In [ ]:
# Playground for RDS internal functions
# One cell per function call (run top-down).

## 1. 기본 설정

In [3]:
import os
from typing import Dict, List, Optional
import psycopg2
import requests

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    load_dotenv = None

def load_env():
    if load_dotenv:
        env_path = os.path.abspath(os.path.join(
            os.getcwd(), 'backend', '.env'
        ))
        if os.path.exists(env_path):
            load_dotenv(env_path)

def get_db_config() -> Dict[str, str]:
    return {
        'host': os.getenv('DB_HOST', 'localhost'),
        'port': os.getenv('DB_PORT', '5432'),
        'dbname': os.getenv('DB_NAME', 'ddoksori'),
        'user': os.getenv('DB_USER', 'postgres'),
        'password': os.getenv('DB_PASSWORD', 'postgres'),
    }

def get_embed_api_url() -> str:
    return os.getenv('EMBED_API_URL', 'http://localhost:8001/embed')

def embed_query(query: str) -> List[float]:
    use_openai = os.getenv('USE_OPENAI_EMBEDDING', 'false').lower() == 'true'
    has_openai_key = bool(os.getenv('OPENAI_API_KEY'))
    if use_openai or has_openai_key:
        try:
            from openai import OpenAI
        except ImportError as exc:
            raise ImportError('openai package is required for OpenAI embeddings. Install it with: pip install openai') from exc
        model = os.getenv('EMBEDDING_MODEL', 'text-embedding-3-large')
        dimensions = int(os.getenv('EMBEDDING_DIMENSIONS', '1536'))
        client = OpenAI()
        response = client.embeddings.create(model=model, input=[query], dimensions=dimensions)
        return response.data[0].embedding
    response = requests.post(get_embed_api_url(), json={'texts': [query]}, timeout=10)
    response.raise_for_status()
    return response.json()['embeddings'][0]

load_env()


# 1) search_similar_chunks

In [5]:
query = 'dmdkdkdkdkdkk'
query_embedding = embed_query(query)

with psycopg2.connect(**get_db_config()) as conn:
    with conn.cursor() as cur:
        cur.execute("""
        SELECT * FROM search_similar_chunks(
            %s::vector, %s, %s, %s, %s, %s
        )
        """, (
            query_embedding,
            'case',  # filter_dataset
            None,  # filter_category
            None,  # filter_law_name
            None,  # filter_year
            5,     # result_limit
        ))
        results = cur.fetchall()

for i, row in enumerate(results, 1):
    print(f'[{i}] sim={row[3]:.4f} id={row[0]}')
    print(f'text_len={len(row[2])}')
    print(row[2])
    print('-' * 60)


[1] sim=0.2906 id=crawl_semantic_조정_239_judgment_3
text_len=26
비자대행계약 불이행에 따른 손해배상 요구

5.
------------------------------------------------------------
[2] sim=0.2906 id=crawl_semantic_조정_2423_judgment_2
text_len=34
중고차 성능 및 사고내역 허위고지에 따른 손해배상 요구

5.
------------------------------------------------------------
[3] sim=0.2906 id=crawl_semantic_조정_1042_judgment_5
text_len=38
자궁경부암 수술 후 급성신부전으로 사망함에 따른 손해배상 요구

5.
------------------------------------------------------------
[4] sim=0.2906 id=crawl_semantic_조정_1577_judgment_2
text_len=36
간세포암 고주파 치료 중 대장 천공 발생에 따른 보상 요구

5.
------------------------------------------------------------
[5] sim=0.2906 id=crawl_semantic_조정_300_judgment_2
text_len=31
조직검사 중 둔부 흉터 발생에 따른 손해배상 요구

5.
------------------------------------------------------------


# 2) search_hybrid

In [ ]:
query = '??? ??'
query_embedding = embed_query(query)

with psycopg2.connect(**get_db_config()) as conn:
    with conn.cursor() as cur:
        cur.execute("""
        SELECT * FROM search_hybrid(
            %s::vector, %s, %s, %s, %s
        )
        """, (
            query_embedding,
            5,     # law_limit
            5,     # case_limit
            None,  # filter_category
            None,  # filter_year
        ))
        results = cur.fetchall()

results[:3]


# 3) search_with_keywords

In [ ]:
query = '??? ??'
keyword = '???'
query_embedding = embed_query(query)

with psycopg2.connect(**get_db_config()) as conn:
    with conn.cursor() as cur:
        cur.execute("""
        SELECT * FROM search_with_keywords(
            %s::vector, %s, %s
        )
        """, (
            query_embedding,
            keyword,
            10,  # result_limit
        ))
        results = cur.fetchall()

results[:3]


# 4) search_bm25

In [ ]:
query_text = '??? ??'

with psycopg2.connect(**get_db_config()) as conn:
    with conn.cursor() as cur:
        cur.execute("""
        SELECT * FROM search_bm25(
            %s, %s, %s, %s, %s
        )
        """, (
            query_text,
            None,  # filter_dataset
            None,  # filter_category
            None,  # filter_document_type
            20,    # result_limit
        ))
        results = cur.fetchall()

results[:3]


# 5) search_hybrid_rrf

In [6]:
query_text = '테스트 테스트 테스트'
query_embedding = embed_query(query_text)

with psycopg2.connect(**get_db_config()) as conn:
    with conn.cursor() as cur:
        cur.execute("""
        SELECT * FROM search_hybrid_rrf(
            %s, %s::vector, %s, %s, %s, %s, %s, %s
        )
        """, (
            query_text,
            query_embedding,
            None,  # filter_dataset
            None,  # filter_category
            None,  # filter_document_type
            None,  # filter_year
            10,    # result_limit
            60,    # rrf_k
        ))
        results = cur.fetchall()

results[:3]


[('상법_제530조의8',
  'law_guide',
  '제530조의8()\n삭제 <2015. 12. 1.>',
  0.01639344262295082,
  0.0,
  0.391152358582476,
  None,
  None,
  None,
  2025,
  {'장': '제4장 주식회사',
   '절': '제11절 회사의 분할',
   '편': '제3편 회사',
   'keywords': [],
   '시행일': '2025-07-22',
   'created_at': '2026-01-19T16:47:42.807647',
   'text_length': 27,
   '법령번호': '법률 제20991호',
   '법령유형': '법률',
   '위임대상': [],
   '조문번호': '제530조의8',
   '조문제목': '',
   '준용조문': [],
   '참조조문': [],
   'example_cases': [],
   '원문_조문': '제530조의8',
   'hierarchy_path': '제3편 회사 > 제4장 주식회사 > 제11절 회사의 분할 > 제530조의8 '}),
 ('crawl_semantic_조정_1078_case_summary_1',
  'case',
  '일방적으로 주문 취소된 한우 2세트 보상요구\n\n가. 신청인은 2015. 3. 20. 피신청인 2로부터 피신청인 1이 수입하는 자동차를 매수{계약금액: 3,700,000원(부가세 포함), 등록 및 취득세 등 제반비용: 2,400,000원, 이하 \'이 사건 매매계약\'이라 한다}하였고, 피신청인 2는 같은 해 3. 30. 별지 자동차 표시 기재 자동차(이하 \'이 사건 자동차\'라 한다)에 대하여 이 사건 매매계약을 원인으로 한 신청인 명의로의 자동차이전등록절차를 이행한 후 신청인에게 이 사건 자동차를 인도하려고 하였으나, 신청인이 이 사건 자동차의 주행거리가 1,000km가 넘고 내부 비닐이 전부 제거되어 있는 점 등을 이유로 그 인수를 거부하였다.나. 신청인은 2015. 